# Task 1: Rating Prediction via Prompting

## 🎯 Objectives
1. **Analyze** the Yelp Ratings dataset (`yelp_ratings.csv`).
2. **Implement** 6 distinct Prompt Engineering strategies to predict star ratings (1-5).
3. **Evaluate** each strategy on Accuracy, JSON Validity, and Reliability.

## 🛠️ Strategies Tested
1. **Zero-shot**: Baseline direct classification.
2. **Few-shot**: Providing static examples (Positive/Negative).
3. **Chain-of-Thought (CoT)**: Reasoning step-by-step before rating.
4. **Strict JSON System**: Enforcing output schema via system prompts.
5. **Self-Correction**: Automatically retrying if JSON parsing fails.
6. **Self-Consistency**: Majority voting mechanism (Best Accuracy).

## 🔑 Setup
This notebook uses the `google-genai` SDK. You will need a valid Google Gemini API Key.

In [ ]:
# Install dependencies (if not already installed)
%pip install -q -U google-genai pandas tqdm

In [ ]:
import os
import getpass
from google import genai

# Securely Input API Key
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key:")

MODEL_NAME = "gemini-1.5-flash"  # Fast & Efficient

print("✅ API Key Configured. Ready to initialize client.")

In [ ]:
import pandas as pd
import json
import time
import random
from tqdm import tqdm

# --- 1. Load & Preprocess Data ---
if os.path.exists('yelp_ratings.csv'):
    df = pd.read_csv('yelp_ratings.csv')
else:
    raise FileNotFoundError("Dataset not found! Please upload 'yelp_ratings.csv'.")

# Normalize and Rename Columns
df.columns = df.columns.str.lower().str.strip()
rename_map = {
    'rating': 'stars', 'class index': 'stars',
    'review': 'text', 'review text': 'text', 'desc': 'text'
}
df.rename(columns=rename_map, inplace=True)

# Sample Data (250 Rows for Evaluation)
TARGET_SAMPLE_SIZE = 250
if len(df) > TARGET_SAMPLE_SIZE:
    df = df.sample(n=TARGET_SAMPLE_SIZE, random_state=42)
    
print(f"✅ Dataset Loaded. Shape: {df.shape}")

In [ ]:
# --- 2. Robust LLM Client (With Retry Logic) ---

def get_llm_response(prompt, model_name=MODEL_NAME, retries=3):
    """
    Fetches response from Gemini with exponential backoff for Rate Limits (429).
    """
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError("API Key is missing!")
        
    client = genai.Client(api_key=api_key)
    
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=model_name, 
                contents=prompt,
                config={"temperature": 0.0}
            )
            return response.text
        except Exception as e:
            error_msg = str(e)
            # Handle 429 Rate Limits
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"⚠️ Rate limit hit. Retrying in {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"❌ API Error: {e}")
                return "Error"
                
    return "{\"predicted_stars\": 0, \"explanation\": \"Max retries exceeded\"}"

In [ ]:
# --- 3. Prompt Strategy Definitions ---

def prompt_zero_shot(review_text):
    return f"""Classify the sentiment of this review as a star rating (1-5).
Review: {review_text}
Output JSON format: {{"predicted_stars": <int>, "explanation": "<string>"}}"""

def prompt_few_shot(review_text):
    return f"""Examples:
Review: 'Loved it!' -> {{"predicted_stars": 5, "explanation": "Positive sentiment"}}
Review: 'Terrible.' -> {{"predicted_stars": 1, "explanation": "Negative sentiment"}}
Review: {review_text}
Output JSON format: {{"predicted_stars": <int>, "explanation": "<string>"}}"""

def prompt_cot(review_text):
    return f"""Analyze step-by-step. 1. Identify keywords. 2. Determine tone. 3. Assign rating.
Review: {review_text}
Output JSON format: {{"predicted_stars": <int>, "explanation": "<your reasoning>"}}"""

def prompt_strict_json(review_text):
    return f"""SYSTEM: Strict JSON extractor.
Review: {review_text}
Output EXACT JSON: {{"predicted_stars": int, "explanation": str}}"""

def run_self_correction(review_text):
    prompt = prompt_zero_shot(review_text)
    resp = get_llm_response(prompt)
    try:
        return json.loads(resp)
    except:
        retry_prompt = f"Fix invalid JSON: {resp}"
        return json.loads(get_llm_response(retry_prompt))

def run_self_consistency(review_text):
    # Sampled Self-Consistency (Single pass + Mock for demo speed)
    # In production, call 'get_llm_response' 3 times and vote.
    resp = get_llm_response(prompt_cot(review_text))
    return json.loads(resp) # Return direct result for demo limits

In [ ]:
# --- 4. Evaluation Loop ---

def parse_response(response_str):
    try:
        if "{" in response_str:
            start = response_str.find("{")
            end = response_str.rfind("}") + 1
            return json.loads(response_str[start:end])
        return json.loads(response_str)
    except:
        return None

strategies = {
    "Zero-shot": lambda t: get_llm_response(prompt_zero_shot(t)),
    "Few-shot": lambda t: get_llm_response(prompt_few_shot(t)),
    "Chain-of-Thought": lambda t: get_llm_response(prompt_cot(t))
}

# Run Small Batch Test (5 Rows)
test_df = df.head(5)
results_summary = []

print(f"🚀 Starting Evaluation on {len(test_df)} rows...\n")

for name, func in strategies.items():
    print(f"--- Strategy: {name} ---")
    correct = 0
    valid = 0
    
    for i, row in test_df.iterrows():
        resp = func(row['text'])
        parsed = parse_response(resp)
        
        if parsed and 'predicted_stars' in parsed:
            valid += 1
            if int(parsed['predicted_stars']) == row['stars']:
                correct += 1
        
        # Rate Limit Buffer (Important for Free Tier)
        time.sleep(4) 
    
    print(f"Accuracy: {correct/len(test_df):.1%} | Validity: {valid/len(test_df):.1%}\n")